# Heavy Rail and Light Rail Data is determined not by this data but rather from MBTA's "MBTA Monthly Ridership By Mode and Line" Data from Open Data Portal

In [ ]:
# Cell 0: read MBTA CSV
from pathlib import Path
import pandas as pd

path = Path("MBTA_Monthly_Ridership_By_Mode_and_Line.csv")
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")


# df = pd.read_csv(path, low_memory=False)

# df = df[(df['month_of_service'] >= '2023-09-01') & (df['month_of_service'] <= '2023-11-30')]

# print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns")
# df.head()

Loaded 0 rows and 7 columns


,month_of_service,daytype,daycount,route_or_line,ridership_total,ridership_average,ObjectId


In [4]:
# reload full CSV and filter for Sep/Oct 2024
df = pd.read_csv(path, low_memory=False)

# ensure month_of_service is datetime (handles strings like "2025/06/01 04:00:00+00")
df['month_of_service'] = pd.to_datetime(df['month_of_service'], errors='coerce', utc=True)

# select desired columns and filter for Sept & Oct 2024
cols = ['month_of_service', 'daytype', 'daycount', 'route_or_line',
    'ridership_total', 'ridership_average', 'ObjectId']

mask = (df['month_of_service'].dt.year == 2024) & (df['month_of_service'].dt.month.isin([9, 10]))
sep_oct_2024 = df.loc[mask, cols].reset_index(drop=True)

print(f"Rows for Sep/Oct 2024: {len(sep_oct_2024)}")
sep_oct_2024.head()

Rows for Sep/Oct 2024: 104


,month_of_service,daytype,daycount,route_or_line,ridership_total,ridership_average,ObjectId
0,2024-10-01 04:00:00+00:00,Weekday,22,Blue Line,1175590,53436,417
1,2024-10-01 04:00:00+00:00,Weekday,22,Orange Line,2409889,109540,418
2,2024-10-01 04:00:00+00:00,Weekday,22,Red Line,3208229,145829,419
3,2024-10-01 04:00:00+00:00,Weekday,22,Heavy Rail,6793708,308805,420
4,2024-10-01 04:00:00+00:00,Weekday,22,Green Line,2666399,121200,421


In [6]:
# filter Weekday rows and aggregate ridership_total and daycount by route_or_line
weekday_data = sep_oct_2024[sep_oct_2024['daytype'] == 'Weekday']

weekday_grouped = (
    weekday_data.groupby('route_or_line', as_index=False)[['ridership_total', 'daycount']]
    .sum()
    .sort_values('ridership_total', ascending=False)
)

print(f"Aggregated rows: {len(weekday_grouped)}")
weekday_grouped

Aggregated rows: 13


,route_or_line,ridership_total,daycount
0,All Bus,13974165,42
6,Heavy Rail,12786827,42
2,Bus,12576381,42
10,Red Line,5670956,42
7,Light Rail,5194422,42
5,Green Line,5006046,42
9,Orange Line,4877865,42
3,Commuter Rail,4800418,43
1,Blue Line,2238006,42
11,Silver Line,1397786,42


In [7]:
# add weekday_average column to weekday_grouped (ridership_total per day)
weekday_grouped['weekday_average'] = weekday_grouped['ridership_total'] / weekday_grouped['daycount']

# show result
weekday_grouped

,route_or_line,ridership_total,daycount,weekday_average
0,All Bus,13974165,42,332718.214286
6,Heavy Rail,12786827,42,304448.261905
2,Bus,12576381,42,299437.642857
10,Red Line,5670956,42,135022.761905
7,Light Rail,5194422,42,123676.714286
5,Green Line,5006046,42,119191.571429
9,Orange Line,4877865,42,116139.642857
3,Commuter Rail,4800418,43,111637.627907
1,Blue Line,2238006,42,53285.857143
11,Silver Line,1397786,42,33280.619048


In [5]:
# retrieve distinct route_or_line for Weekday from the Sep/Oct 2024 subset
weekday_routes = (
    sep_oct_2024.loc[sep_oct_2024['daytype'] == 'Weekday', 'route_or_line']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(f"Found {len(weekday_routes)} distinct weekday routes/lines:")
print(weekday_routes.tolist())

# optional: expose as a dataframe for further use/inspection
weekday_routes_df = weekday_routes.to_frame(name='route_or_line')
weekday_routes_df

Found 13 distinct weekday routes/lines:
['All Bus', 'Blue Line', 'Bus', 'Commuter Rail', 'Ferry', 'Green Line', 'Heavy Rail', 'Light Rail', 'Mattapan Line', 'Orange Line', 'Red Line', 'Silver Line', 'The RIDE']


,route_or_line
0,All Bus
1,Blue Line
2,Bus
3,Commuter Rail
4,Ferry
5,Green Line
6,Heavy Rail
7,Light Rail
8,Mattapan Line
9,Orange Line


In [ ]:
df.to_csv("MBTA_Heavy_Light_Rail_Ridership_Fall2023.csv", index=False)

In [ ]:
# aggregate boardings by route_id and day_type_name
agg_boardings = (
    df.groupby(['route_id', 'day_type_name'], as_index=False)['average_ons']
      .sum()
      .rename(columns={'average_ons': 'total_boardings'})
)

# sort for easier inspection (highest totals first)
agg_boardings = agg_boardings.sort_values(['route_id', 'total_boardings'], ascending=[True, False])


In [ ]:
agg_boardings = agg_boardings[agg_boardings['day_type_name'] == 'Weekday']

In [ ]:
agg_boardings

,route_id,day_type_name,total_boardings
2,Blue,Weekday,63193
5,Green,Weekday,102947
8,Orange,Weekday,102362
11,Red,Weekday,129171


In [ ]:
agg_boardings[['route_id', 'total_boardings']]
agg_boardings.to_csv("rail_route_weekday_boardings_aggregate.csv", index=False)

In [ ]:
total_boardings_sum = agg_boardings['total_boardings'].sum()
print(f"Sum of total_boardings: {total_boardings_sum:,.1f}")

Sum of total_boardings: 397,673.0


In [ ]:
# Scale total_boardings so that their sum matches 111,755
target_sum = 111_755

scale_factor = target_sum / total_boardings_sum

agg_boardings_scaled = agg_boardings.copy()
agg_boardings_scaled['total_boardings'] = agg_boardings_scaled['total_boardings'] * scale_factor

print(f"Scaled sum: {agg_boardings_scaled['total_boardings'].sum():,.1f}")
agg_boardings_scaled.head()

Scaled sum: 111,755.0


,route_id,day_type_name,total_boardings
2,Blue,Weekday,17758.645206
5,Green,Weekday,28930.407609
8,Orange,Weekday,28766.009535
11,Red,Weekday,36299.937650


In [ ]:
agg_boardings_scaled

,route_id,day_type_name,total_boardings
2,Blue,Weekday,17758.645206
5,Green,Weekday,28930.407609
8,Orange,Weekday,28766.009535
11,Red,Weekday,36299.937650


In [ ]:
# aggregate statistics on agg_boardings.total_boardings (weekday)
tb = agg_boardings['total_boardings']

agg_stats = pd.Series({
    'count': tb.count(),
    'sum': tb.sum(),
    'mean': tb.mean(),
    'median': tb.median(),
    'std': tb.std(),
    'min': tb.min(),
    'max': tb.max()
})
print(agg_stats)

# re-aggregate by route_id (defensive) and sort
per_route = agg_boardings.groupby('route_id', as_index=False)['total_boardings'].sum()
per_route = per_route.sort_values('total_boardings', ascending=False)

# top / bottom routes
print("\nTop 10 routes by weekday total_boardings:")
print(per_route.head(10))

print("\nBottom 10 routes by weekday total_boardings:")
print(per_route.tail(10))

# add percent and cumulative share, save results
per_route['pct_share'] = per_route['total_boardings'] / per_route['total_boardings'].sum()
per_route['cum_share'] = per_route['pct_share'].cumsum()
per_route.to_csv("weekday_boardings_per_route_with_shares.csv", index=False)

# quick bar plot of top 20 routes (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    per_route.set_index('route_id')['total_boardings'].head(20).plot(
        kind='bar', figsize=(10,4), title='Top 20 routes — weekday total boardings'
    )
    plt.ylabel('total_boardings')
    plt.tight_layout()
except Exception:
    pass

count          4.000000
sum       397673.000000
mean       99418.250000
median    102654.500000
std        27194.439448
min        63193.000000
max       129171.000000
dtype: float64

Top 10 routes by weekday total_boardings:
  route_id  total_boardings
3      Red           129171
1    Green           102947
2   Orange           102362
0     Blue            63193

Bottom 10 routes by weekday total_boardings:
  route_id  total_boardings
3      Red           129171
1    Green           102947
2   Orange           102362
0     Blue            63193
